# 🎤 INSTRUCTOR — AI Demo notebook
**Goal:** model the full workflow on 3 tasks while students watch (and follow along in their copy `02_STUDENT_ai_demo_followalong`).

**How to run it (~35–40 min):**
1. Read the workflow (5 steps) — 3 min.
2. For each task: read the task → students predict → paste the **weak prompt** into your AI tool of choice and show the answer → paste the **strong prompt** → run the **verification** cell → name the lesson.
3. Use whatever AI you like (Claude, ChatGPT, Gemini…). **Answers differ from run to run** — the "what to look for" boxes tell you what to check, and the verification cells hold the ground truth.

> I deliberately did *not* paste fake AI transcripts: do it live; it's more convincing, and real mistakes are the best teacher. If the AI gets a task right, that's fine: the lesson is *"how do I know?"*

In [2]:
#@title ▶️ Setup — run this cell first { display-mode: "form" }
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
rng = np.random.default_rng(42)     # fixed seed → everyone sees the same random numbers
print("✅ Setup complete")

✅ Setup complete


## The 5-step AI workflow  **U-P-R-V-R**
| Step | What you do | Why |
|---|---|---|
| **1 · Understand** | Restate the problem in your own words. List what's *given* and what's *asked*. | If you can't say it, you can't check it |
| **2 · Prompt** | Give context + all numbers + the exact question. Ask for: assumptions, steps, formula **and** Python code | Vague in → vague out |
| **3 · Run** | Run the code yourself. Never copy results you didn't execute | AI can describe code that doesn't work |
| **4 · Verify** | Simulation · sanity bounds (is 0 ≤ p ≤ 1? plausible size?) · try a simpler case · ask the AI to *attack its own answer* | The step that makes you a data scientist |
| **5 · Reflect** | Log: what did AI get right/wrong? What did you change? | You learn the failure patterns |

### ✍️ Anatomy of a strong prompt
```
ROLE/CONTEXT : I'm a data science student learning probability.
PROBLEM      : <paste the full problem with ALL numbers>
CONSTRAINTS  : Use Python (numpy/scipy). State your assumptions.
DELIVERABLE  : 1) the formula  2) the numeric answer  3) code  4) a simulation that checks it
CHECK        : Tell me one way your answer could be wrong.
```

---
# Demo Task 1 · The false-alarm test (Bayes)

> **Task:** A factory machine is faulty **2 %** of the time. A sensor flags 90 % of faulty runs, and also wrongly flags **8 %** of healthy runs.
> The sensor flags a run. **What's the probability the machine is actually faulty?**

### 🔮 PREDICT (answer in chat *before* running the next cell)

**Gut check in chat** — a number between 0 and 100 %. *(Don't compute.)*

### ❌ Weak prompt
```
what is the chance the machine is faulty if the sensor flags it
```
### ✅ Strong prompt *(copy → AI)*
```
I'm learning Bayes' theorem. A machine is faulty 2% of the time. A sensor flags 90% of faulty runs
and wrongly flags 8% of healthy runs. The sensor flags a run.
1) Write the quantities as P(...) notation (prior, likelihood, false positive rate).
2) Compute P(faulty | flagged) with Bayes' theorem, showing P(flagged).
3) Give Python code, and a simulation of 1,000,000 runs that checks the result.
4) Tell me one common mistake people make on this kind of problem.
```

> 👀 **What to look for:** AI almost always gets Bayes problems right, so the lesson here is the *prompt structure* and the *verification*: the AI should mention that P(flagged) must include the false alarms from the 98 % healthy runs. Ask it: "what would your answer be if the machine were faulty 20 % of the time?" → shows the base-rate effect.

In [9]:
# ---- our verification (ground truth) ----
prior, sens, fpr = 0.02, 0.90, 0.08
p_flag = sens * prior + fpr * (1 - prior)
exact = sens * prior / p_flag

n = 1_000_000
faulty = rng.random(n) < prior
flagged = np.where(faulty, rng.random(n) < sens, rng.random(n) < fpr)
print(f"exact (Bayes)  : {exact:.4f}")
print(f"simulation     : {faulty[flagged].mean():.4f}")

exact (Bayes)  : 0.1867
simulation     : 0.1856


> 🎤 LESSON: "The AI's formula and our simulation agree → now I **trust** it. Agreement of two independent methods is the cheapest quality check you have." (Exact ≈ 0.186.)

---
# Demo Task 2 · The call centre (choosing the distribution)

> **Task:** A call centre receives on average **4 calls per minute**, at random times. **What's the probability of getting *at least 7* calls in a given minute?**

### 🔮 PREDICT (answer in chat *before* running the next cell)

Which **distribution** would you use (Binomial / Poisson / Normal)? And do you expect the answer to be closer to 1 %, 10 % or 40 %?

### ✅ Strong prompt *(copy → AI)*
```
A call centre gets an average of 4 calls per minute at random times.
1) Which probability distribution models the number of calls in a minute, and WHY (state the assumptions)?
2) Compute P(X >= 7) with scipy. Be explicit about whether you use cdf(6) or cdf(7) and why.
3) Verify with a simulation of 500,000 minutes.
```

> 👀 **What to look for (common AI traps):**
> • Off-by-one: P(X ≥ 7) = 1 − cdf(**6**). Some answers use cdf(7), which computes P(X > 7).
> • Wrong distribution (Binomial without an n; Normal approximation giving a slightly different answer).
> 👉 If the AI got it right, ask: "Now P(X > 7)?" and ask students whether the number should change — it should.

In [16]:
lam = 4
right = 1 - stats.poisson.cdf(6, lam)       # P(X >= 7)
wrong = 1 - stats.poisson.cdf(7, lam)       # P(X >= 8)  <- the classic off-by-one
sim = (rng.poisson(lam, 500_000) >= 7).mean()
print(f"P(X >= 7) exact      : {right:.4f}")
print(f"P(X >= 7) simulation : {sim:.4f}")
print(f"(off-by-one version) : {wrong:.4f}   <- this is actually P(X >= 8)")

P(X >= 7) exact      : 0.1107
P(X >= 7) simulation : 0.1104
(off-by-one version) : 0.0511   <- this is actually P(X >= 8)


> 🎤 LESSON: "Same code, one number different, a wrong answer that *looks* just as confident. A simulation caught it — **always verify the boundary (≥ vs >).**"

---
# Demo Task 3 · Monty Hall with a twist (read the problem carefully!)

> **Task:** There are 3 doors: 1 car, 2 goats. You pick door 1. The host **does not know** where the car is. He opens one of the other doors **at random** — and it happens to show a **goat**. You may switch to the remaining door.
> **What is the probability of winning the car if you switch?**

### 🔮 PREDICT (answer in chat *before* running the next cell)

Classic Monty Hall says 'switch → 2/3'. **Does the answer change here?** A) still 2/3 &nbsp; B) 1/2 &nbsp; C) 1/3

### ❌ Weak prompt → AI pattern-matches the famous version
```
monty hall problem, should I switch?
```
### ✅ Strong prompt *(copy → AI)*
```
3 doors, one car. I pick door 1. The host does NOT know where the car is and opens one of the other
two doors at random. He opens door 3 and it shows a goat.
1) Is this the same as the classic Monty Hall problem? Explain precisely what differs.
2) Compute P(car behind door 2 | host randomly opened door 3 and showed a goat) using Bayes.
3) Simulate it: simulate many games, KEEP ONLY the games where the host's random door shows a goat,
   and estimate the win rate of switching.
```

> 👀 **What to look for:** a *weak* prompt usually gets the classic answer, 2/3 — right for the classic problem, wrong here. The strong prompt forces the AI to notice "host does not know".

In [23]:
n = 1_000_000
car = rng.integers(1, 4, n)                         # door hiding the car (you always pick door 1)
# --- classic: host KNOWS and always opens a goat door ---
switch_classic_win = (car != 1)                     # switching wins iff your first pick was wrong
print("classic Monty Hall, switch wins:", round(switch_classic_win.mean(), 4))

# --- twist: host opens door 2 or 3 at random, may reveal the car ---
opened = rng.integers(2, 4, n)
shows_goat = opened != car                          # keep only games where he happened to reveal a goat
other_door = np.where(opened == 2, 3, 2)
switch_win = (car == other_door)
print("twist (host doesn't know), switch wins | goat shown:", round(switch_win[shows_goat].mean(), 4))
print("share of games discarded because the car was revealed:", round(1 - shows_goat.mean(), 3))

classic Monty Hall, switch wins: 0.6673
twist (host doesn't know), switch wins | goat shown: 0.5008
share of games discarded because the car was revealed: 0.334


> 🎤 LESSON: "The *information-generating process* matters. Same picture, different rules → different probability. AI answers the famous problem it recognises, not necessarily the one you asked. **Precise problem statements + verification** beat any magic prompt." (Twist answer: 1/2.)

---
## 🧾 Your AI log (every task, every time)
| Field | Example |
|---|---|
| **Prompt I used** | (paste) |
| **What the AI answered** | (summary + number) |
| **How I verified** | simulation / simpler case / second method |
| **What was wrong or missing** | e.g. off-by-one, wrong distribution |
| **What I changed** | |

## 🎓 Take-aways
1. **Give AI the whole problem with numbers**, ask for assumptions, formula, code *and* a simulation.
2. **Never trust a number you haven't verified** — simulation is your best friend in probability.
3. **Watch the usual suspects:** P(A|B) vs P(B|A), ≥ vs >, wrong distribution, the base rate, the *exact* wording.
4. **You are accountable for the answer, not the AI.** Log what you checked.

➡️ Now it's your turn: open **`03_STUDENT_ai_tasks`**.